# 1.3 — Глубокий анализ параметров CRR

**Папка 1, подноутбук 3.** Кривая сопротивления разжижению строится физической моделью
(model.py): **CRR(N) = β / N^(1 − α)**, где α и β выводятся из физико-механических свойств
грунта через цепочку поправок к базовому уровню CRR15. Здесь анализируются именно те
параметры, которые **строят CRR**: их распределения, зависимость от свойств грунта,
вклад каждого физического фактора и сами кривые CRR(N) по типам грунта.

## Окружение и загрузка артефакта

In [2]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

from liquefaction_ai.physics import crr_curve
from liquefaction_ai.viz import box_grid, correlation_heatmap, heatmap, histogram_grid, lines, scatter
from liquefaction_ai import load_population_artifact
from liquefaction_ai.constants import (FEATURE_AXIS_LABELS_EN, LOAD_DISPLAY_NAMES_EN,
                                       LOAD_NAMES, SOIL_DISPLAY_NAMES_EN, SOIL_NAMES)
from liquefaction_ai.data.grainsize import FRACTION_BOUNDS, FRACTION_KEYS, TYPE_GROUND_NAMES_EN

population, config = load_population_artifact(DATA_DIR)
meta = population["meta"].copy()
meta["log10_permeability"] = np.log10(meta["permeability"])
meta["soil_en"] = meta["soil_type"].map(SOIL_DISPLAY_NAMES_EN)
meta["load_en"] = meta["load_mode"].map(LOAD_DISPLAY_NAMES_EN)
# Совместимость с реальными данными: синтетические поля заменяем наблюдаемым прокси (пик PPR)
for _col in ["damage_max_true"]:
    if _col not in meta.columns:
        meta[_col] = meta["PPR_max_true"]
soil_order = list(SOIL_NAMES); load_order = list(LOAD_NAMES)
soil_present = [s for s in soil_order if (meta["soil_type"] == s).any()]
print("Loaded scenarios:", len(meta))

Loaded scenarios: 666


## Распределения параметров кривой CRR

`crr_ref` — уровень CRR при 15 циклах (CRR15), `crr_alpha`/`crr_betta` — параметры кривой,
`crr_cycle_slope` — показатель циклической деградации s = 1 − α.

In [3]:
cols = ["crr_ref", "crr_alpha", "crr_betta", "crr_cycle_slope"]
histogram_grid(meta, columns=cols, titles=[FEATURE_AXIS_LABELS_EN.get(c, c) for c in cols],
               n_cols=2, title="Distributions of CRR curve parameters",
               save=SAVE_FIGS, fig_id="1_3_crr_param_distributions").show()

[save_figure] PNG для '1_3_crr_param_distributions' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Параметры CRR по типам грунта

In [4]:
cols = ["crr_ref", "crr_alpha", "crr_betta"]
box_grid(meta, value_cols=cols, group_col="soil_type", group_order=soil_present,
         group_labels=SOIL_DISPLAY_NAMES_EN, titles=[FEATURE_AXIS_LABELS_EN.get(c, c) for c in cols],
         n_cols=3, title="CRR parameters within soil types",
         save=SAVE_FIGS, fig_id="1_3_crr_by_soil").show()

[save_figure] PNG для '1_3_crr_by_soil' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Кривые CRR(N) по типам грунта

Средняя кривая сопротивления для каждого типа грунта на горизонте до 1500 циклов.

In [5]:
N = np.logspace(0, np.log10(1500.0), 60)
series = []
for s in soil_present:
    m = meta["soil_type"].to_numpy() == s
    a = float(meta.loc[m, "crr_alpha"].mean()); b = float(meta.loc[m, "crr_betta"].mean())
    series.append({"x": N, "y": b / N ** (1.0 - a), "name": SOIL_DISPLAY_NAMES_EN[s]})
lines(series, title="Mean CRR(N) resistance curves by soil type",
      xlabel="Number of cycles, N", ylabel="Cyclic resistance ratio, CRR (–)", logx=True,
      save=SAVE_FIGS, fig_id="1_3_crr_curves_by_soil").show()

[save_figure] PNG для '1_3_crr_curves_by_soil' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Вклад физических факторов в CRR

CRR_ref = CRR15(Dr) · Π factor_i. Тепловая карта показывает средний множитель каждого
фактора по типам грунта: значения > 1 повышают сопротивление, < 1 — снижают.

In [6]:
factor_cols = [c for c in meta.columns if c.startswith("crr_f_")]
factor_labels = [c.replace("crr_f_", "") for c in factor_cols]
mat = np.array([[float(meta.loc[meta["soil_type"] == s, fc].mean()) for fc in factor_cols] for s in soil_present])
heatmap(mat, x_labels=factor_labels, y_labels=[SOIL_DISPLAY_NAMES_EN[s] for s in soil_present],
        title="Mean CRR correction factors by soil type", colorscale="RdBu",
        value_fmt=".2f", colorbar_title="factor", height=460,
        save=SAVE_FIGS, fig_id="1_3_crr_factor_heatmap").show()

[save_figure] PNG для '1_3_crr_factor_heatmap' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Чувствительность CRR к свойствам грунта

CRR15 растёт с относительной плотностью Dr; содержание мелких частиц и пластичность
модулируют уровень и наклон кривой.

In [7]:
scatter(meta["D_r"], meta["crr_ref"], color=meta["fines_content"], color_label="Fines content (%)",
        title="CRR15 vs relative density (coloured by fines)", xlabel="Relative density, Dr (–)",
        ylabel="CRR at 15 cycles (–)", save=SAVE_FIGS, fig_id="1_3_crr_vs_dr").show()
scatter(meta["fines_content"], meta["crr_cycle_slope"], color=meta["I_p"], color_label="Plasticity index (%)",
        title="Cycle-degradation slope vs fines (coloured by Ip)", xlabel="Fines content (%)",
        ylabel="Cycle-degradation slope, s (–)", save=SAVE_FIGS, fig_id="1_3_slope_vs_fines").show()

[save_figure] PNG для '1_3_crr_vs_dr' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '1_3_slope_vs_fines' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Корреляция CRR-строящих параметров с CRR15

Какие свойства сильнее всего определяют уровень сопротивления.

In [8]:
crr_inputs = ["D_r", "e", "I_p", "Il", "fines_content", "clay_fraction", "sigma_eff",
              "K0", "Vs1", "static_shear_ratio", "cementation_index", "crr_ref", "crr_alpha"]
crr_inputs = [c for c in crr_inputs if c in meta.columns]
correlation_heatmap(meta[crr_inputs].corr(), title="Correlations among CRR-building parameters",
                    save=SAVE_FIGS, fig_id="1_3_crr_input_correlation").show()

[save_figure] PNG для '1_3_crr_input_correlation' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

Параметры CRR физически связаны со свойствами грунта: видно влияние плотности, мелких
частиц, пластичности, давления и Vs1. Следующий шаг — **1.4 разбиение выборок**.